# Model Drift Detection — Instacart Reorder Prediction

Computes Population Stability Index (PSI) between `gold.training_features` (baseline) and `gold.serving_features` (current) to detect data drift. If PSI exceeds the threshold for any feature, a new SageMaker Training Job is submitted automatically.

**Input:** `AwsDataCatalog.gold.training_features`, `AwsDataCatalog.gold.serving_features` (Iceberg)  
**Output:** Per-feature PSI report printed to console; SageMaker Training Job triggered if drift detected

## 1. Setup

In [ ]:
import boto3
import awswrangler as wr
import numpy as np
from datetime import datetime, timezone

BUCKET_NAME = "instacart-aws-data-ml-eng-project"
ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-results/"
DATABASE = "gold"
REGION = "ap-southeast-2"
SAGEMAKER_ROLE = "arn:aws:iam::606476261726:role/instacart-sagemaker-role"
PSI_THRESHOLD = 0.1

FEATURE_COLUMNS = [
    "user_product_buy_count",
    "user_product_last_order_in_window",
    "user_product_avg_add_to_cart",
    "user_product_orders_since_last_buy",
    "user_window_orders",
    "user_avg_days_between",
    "user_total_orders",
    "user_product_window_reorder_ratio",
    "item_total_sales",
    "item_reorder_rate",
]

print("Setup complete.")

## 2. Load Feature Tables

In [ ]:
cols = ", ".join(FEATURE_COLUMNS)

training_df = wr.athena.read_sql_query(
    f"SELECT {cols} FROM gold.training_features",
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

serving_df = wr.athena.read_sql_query(
    f"SELECT {cols} FROM gold.serving_features",
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

print(f"Training samples: {len(training_df):,}")
print(f"Serving samples:  {len(serving_df):,}")

## 3. Compute PSI

In [ ]:
def compute_psi(baseline: np.ndarray, current: np.ndarray, n_bins: int = 10) -> float:
    eps = 1e-6
    bin_edges = np.linspace(
        min(baseline.min(), current.min()),
        max(baseline.max(), current.max()),
        n_bins + 1,
    )
    baseline_pct = np.histogram(baseline, bins=bin_edges)[0] / len(baseline) + eps
    current_pct  = np.histogram(current,  bins=bin_edges)[0] / len(current)  + eps
    return float(np.sum((current_pct - baseline_pct) * np.log(current_pct / baseline_pct)))


drift_results = []
drift_detected = False

print(f"PSI threshold: {PSI_THRESHOLD}\n")
print(f"{'Column':<45} {'PSI':>8}  Status")
print("-" * 65)

for col in FEATURE_COLUMNS:
    baseline = training_df[col].dropna().values
    current  = serving_df[col].dropna().values

    if len(baseline) == 0 or len(current) == 0:
        print(f"{col:<45} {'N/A':>8}  SKIPPED (empty)")
        continue

    psi = compute_psi(baseline, current)
    drift_results.append({"column": col, "psi": psi})

    if psi > PSI_THRESHOLD:
        drift_detected = True
        status = "DRIFT"
    else:
        status = "OK"

    print(f"{col:<45} {psi:>8.4f}  {status}")

max_psi = max(r["psi"] for r in drift_results) if drift_results else 0.0
print("-" * 65)
print(f"{'Max PSI':<45} {max_psi:>8.4f}")
print(f"\nDrift detected: {drift_detected}")

## 4. Trigger Retrain (if drift detected)

In [ ]:
if not drift_detected:
    print("No significant drift detected. Retrain not required.")
else:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    job_name = f"instacart-reorder-training-drift-{timestamp}"

    sm = boto3.client("sagemaker", region_name=REGION)
    response = sm.create_training_job(
        TrainingJobName=job_name,
        RoleArn=SAGEMAKER_ROLE,
        AlgorithmSpecification={
            "TrainingImage": "683313688378.dkr.ecr.ap-southeast-2.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
            "TrainingInputMode": "File",
        },
        HyperParameters={
            "bucket-name": BUCKET_NAME,
            "model-name": "instacart-reorder-model",
            "mlflow-tracking-uri": "arn:aws:sagemaker:ap-southeast-2:606476261726:mlflow-tracking-server/instacart-mlflow",
            "pr-auc-gate": "0.35",
            "sample-users": "10000",
            "sagemaker_program": "train.py",
            "sagemaker_submit_directory": f"s3://{BUCKET_NAME}/scripts/",
        },
        OutputDataConfig={"S3OutputPath": f"s3://{BUCKET_NAME}/sagemaker-output/"},
        ResourceConfig={
            "InstanceType": "ml.m5.xlarge",
            "InstanceCount": 1,
            "VolumeSizeInGB": 20,
        },
        StoppingCondition={"MaxRuntimeInSeconds": 3600},
    )

    print(f"Retrain job submitted: {job_name}")
    print(f"HTTP status: {response['ResponseMetadata']['HTTPStatusCode']}")
    print(f"\nCheck status:")
    print(f"  aws sagemaker describe-training-job --training-job-name {job_name} --region {REGION} --query TrainingJobStatus --output text")
    print(f"\nStream logs:")
    print(f"  aws logs tail /aws/sagemaker/TrainingJobs --log-stream-name-prefix {job_name} --follow")

## 5. Verify

In [ ]:
import pandas as pd

summary = pd.DataFrame(drift_results).sort_values("psi", ascending=False).reset_index(drop=True)
summary["drifted"] = summary["psi"] > PSI_THRESHOLD
summary